# 🧹 Limpeza: Downloads/MBA vs Documents/MBA---Data-Engineering

Compara os dois diretórios pelo **nome do arquivo** (não pelo caminho completo)  
e **deleta da pasta Downloads** tudo que já existe na pasta Documents.

> ⚠️ **Rode a célula de simulação antes de executar a exclusão real!**

## ⚙️ Configuração

In [1]:
import os
from pathlib import Path

ORIGEM     = Path(r"C:\Users\JOAO PC\Downloads\MBA")
REFERENCIA = Path(r"C:\Users\JOAO PC\Documents\MBA---Data-Engineering")

print(f"Origem    : {ORIGEM}  →  existe? {ORIGEM.exists()}")
print(f"Referência: {REFERENCIA}  →  existe? {REFERENCIA.exists()}")

Origem    : C:\Users\JOAO PC\Downloads\MBA  →  existe? True
Referência: C:\Users\JOAO PC\Documents\MBA---Data-Engineering  →  existe? True


## 🌳 Mapear arquivos por nome

In [2]:
def mapear_arquivos(raiz: Path) -> dict:
    """
    Retorna dict: nome_do_arquivo -> lista de Paths absolutos.
    Lista porque pode haver arquivos com mesmo nome em pastas diferentes.
    """
    mapa = {}
    for item in raiz.rglob("*"):
        if item.is_file():
            mapa.setdefault(item.name, []).append(item)
    return mapa

mapa_origem     = mapear_arquivos(ORIGEM)
mapa_referencia = mapear_arquivos(REFERENCIA)

print(f"Arquivos em Downloads/MBA              : {sum(len(v) for v in mapa_origem.values())}")
print(f"Arquivos em Documents/MBA (referência) : {sum(len(v) for v in mapa_referencia.values())}")

Arquivos em Downloads/MBA              : 227
Arquivos em Documents/MBA (referência) : 548


## 🔍 Identificar arquivos em comum (mesmo nome)

In [3]:
nomes_comuns = set(mapa_origem.keys()) & set(mapa_referencia.keys())

# Monta lista de (nome, path_na_origem, paths_na_referencia)
para_deletar = []
for nome in sorted(nomes_comuns):
    for path_orig in mapa_origem[nome]:
        para_deletar.append((nome, path_orig, mapa_referencia[nome]))

print(f"Nomes de arquivo em comum        : {len(nomes_comuns)}")
print(f"Arquivos a deletar da origem     : {len(para_deletar)}")

Nomes de arquivo em comum        : 96
Arquivos a deletar da origem     : 96


## 👀 Simulação (DRY-RUN) — nada é deletado aqui

In [4]:
if not para_deletar:
    print("✅ Nenhum arquivo em comum encontrado.")
else:
    print(f"📋 {len(para_deletar)} arquivo(s) que SERIAM deletados de Downloads/MBA:\n")
    for nome, path_orig, paths_ref in para_deletar:
        print(f"  📄 {nome}")
        print(f"     DELETAR   → {path_orig}")
        for pr in paths_ref:
            print(f"     JÁ EXISTE → {pr}")
        print()

📋 96 arquivo(s) que SERIAM deletados de Downloads/MBA:

  📄 0 -Data Pipeline - Apresentacao.pdf
     DELETAR   → C:\Users\JOAO PC\Downloads\MBA\Data Pipelines\0 -Data Pipeline - Apresentacao.pdf
     JÁ EXISTE → C:\Users\JOAO PC\Documents\MBA---Data-Engineering\fiap_downloads\materiais_aula\Data Pipelines Thiago Nascimento Nogueira\0 -Data Pipeline - Apresentacao.pdf

  📄 0.1-Databricks.pdf
     DELETAR   → C:\Users\JOAO PC\Downloads\MBA\Stream Processing Pipelines\0.1-Databricks.pdf
     JÁ EXISTE → C:\Users\JOAO PC\Documents\MBA---Data-Engineering\fiap_downloads\materiais_aula\Stream Processing Pipelines Rafael Santos Novo Pereira\Aula1210_labs\0.1-Databricks.pdf

  📄 01 - MBA_FIAP_2024_Introdução_NoSQL_columnar_timeseries_databases.pdf
     DELETAR   → C:\Users\JOAO PC\Downloads\MBA\Colunar e Time Series Databases\01 - MBA_FIAP_2024_Introdução_NoSQL_columnar_timeseries_databases.pdf
     JÁ EXISTE → C:\Users\JOAO PC\Documents\MBA---Data-Engineering\fiap_downloads\materiais_aula\Colu

## 🗑️ EXECUÇÃO REAL — deleta os arquivos em comum

> ⚠️ **Só rode essa célula depois de revisar a simulação acima!**

In [5]:
deletados = 0
erros = []

for nome, path_orig, paths_ref in para_deletar:
    try:
        path_orig.unlink()
        print(f"  ✅ deletado | {path_orig.name}")
        print(f"              {path_orig}")
        deletados += 1
    except Exception as e:
        print(f"  ❌ erro     | {path_orig} → {e}")
        erros.append(str(path_orig))

print("\n" + "=" * 60)
print(f"✅ Arquivos deletados : {deletados}")
print(f"❌ Erros             : {len(erros)}")

  ✅ deletado | 0 -Data Pipeline - Apresentacao.pdf
              C:\Users\JOAO PC\Downloads\MBA\Data Pipelines\0 -Data Pipeline - Apresentacao.pdf
  ✅ deletado | 0.1-Databricks.pdf
              C:\Users\JOAO PC\Downloads\MBA\Stream Processing Pipelines\0.1-Databricks.pdf
  ✅ deletado | 01 - MBA_FIAP_2024_Introdução_NoSQL_columnar_timeseries_databases.pdf
              C:\Users\JOAO PC\Downloads\MBA\Colunar e Time Series Databases\01 - MBA_FIAP_2024_Introdução_NoSQL_columnar_timeseries_databases.pdf
  ✅ deletado | 01-Banco Dados Relacional - Transação - Concorrência.pptx
              C:\Users\JOAO PC\Downloads\MBA\Relational Database e Advanced SQL\01-Banco Dados Relacional - Transação - Concorrência.pptx
  ✅ deletado | 03 - Cassandra - Introducao- MBA_FIAP_2024_NoSQL_columnar_timeseries_databases.pdf
              C:\Users\JOAO PC\Downloads\MBA\Colunar e Time Series Databases\03 - Cassandra - Introducao- MBA_FIAP_2024_NoSQL_columnar_timeseries_databases.pdf
  ✅ deletado | 04 - Cassan

## 🧹 Bonus: remover pastas vazias que sobraram na origem

In [6]:
pastas_removidas = 0

# Percorre de baixo para cima (mais profundo primeiro)
for pasta in sorted(ORIGEM.rglob("*"), key=lambda p: len(p.parts), reverse=True):
    if pasta.is_dir():
        try:
            pasta.rmdir()  # só remove se vazia
            print(f"  🗂️  removida | {pasta.relative_to(ORIGEM)}")
            pastas_removidas += 1
        except OSError:
            pass  # ainda tem conteúdo, ignora

print(f"\nPastas vazias removidas: {pastas_removidas}")


Pastas vazias removidas: 0
